# Your own processing configuration: register a profile in Python, select it by name

A *profile* (`FidelityProfile`) names one variant of the UIOOPS source and carries the switches and
reject thresholds that distinguish it. Every field of a profile is a threshold or a behaviour, so
the package keeps one rule: **a profile is described only in Python, with `register_profile()`; a
configuration file may only select a registered profile by name, never describe one.** This
notebook lists the shipped profiles, derives a variant from one by changing a single reject
threshold, registers it, selects it from a TOML file read by the package's config loader, shows a
TOML file that tries to set a threshold being refused, and shows how the chosen profile is
recorded in a product's `.call` stamp. No data are processed.

**Before this:** [03_whole_campaign](03_whole_campaign.ipynb) (Processing a whole campaign). **Next:** [the examples index](../README.md). **Needs:** core package only.

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/2_processing/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

## The shipped profiles

`profiles.SHIPPED` holds the names the package ships; `profiles.PROFILES` is the live registry,
which also holds anything registered in this process. `profiles.profile(name)` returns one.

In [2]:
import dataclasses
from arrayprobe import profiles

print("stock-v3.5 shipped:", "stock-v3.5" in profiles.SHIPPED)
base = profiles.profile("stock-v3.5")
print("base:", base.name)
print("reject thresholds:", base.reject_thresholds)

stock-v3.5 shipped: True
base: stock-v3.5
reject thresholds: RejectConfig(rej_zero_area=True, rej_split=True, rej_hollow=True, shadowed_area=25, max_edge_img_ar=6, max_comp_img_ar=5)


## Derive a variant by changing one field

A `FidelityProfile` is a frozen dataclass, so `dataclasses.replace` builds a copy with chosen
fields changed. Here the variant tightens one reject threshold, the edge-touching aspect-ratio
limit `max_edge_img_ar`, from 6 to 5, and keeps every other field of the base. It needs its own
`name`, which is what a `.call` stamp records, and a `description` saying which source tree it
is; `register_profile()` refuses a profile without either.

In [3]:
mine = dataclasses.replace(
    base,
    name="my-lab-edge5",
    description=("Stock UIOOPS v3.5 with max_edge_img_ar = 5 in calculate_reject_unified.m, "
                 "as edited in my lab's copy of the source tree."),
    reject_thresholds=dataclasses.replace(base.reject_thresholds, max_edge_img_ar=5),
)
changed = [f.name for f in dataclasses.fields(base)
           if f.name not in ("name", "description") and getattr(base, f.name) != getattr(mine, f.name)]
print("fields that differ from the base:", changed)
for f in dataclasses.fields(base.reject_thresholds):
    a, b = getattr(base.reject_thresholds, f.name), getattr(mine.reject_thresholds, f.name)
    print(f"  {f.name:16s} {str(a):>6s} -> {str(b):>6s}" + ("   <- changed" if a != b else ""))

fields that differ from the base: ['reject_thresholds']
  rej_zero_area      True ->   True
  rej_split          True ->   True
  rej_hollow         True ->   True
  shadowed_area        25 ->     25
  max_edge_img_ar       6 ->      5   <- changed
  max_comp_img_ar       5 ->      5


## Register it

`register_profile()` adds the variant to the registry under its name and returns it. A name can be
registered once: a stamp records the name, so a name must keep meaning one variant, and a second
registration under the same name raises `ValueError`. By default a registration declares that its
citations name the v3.1.4 tree (`cites_tree="3.1.4"`), so it is usable with the v3.1.4 engine (not part of this package);
`is_v35_flavoured` says whether the v3.5 engine would accept it.

In [4]:
registered = profiles.register_profile(mine)
print("registered:", registered.name, "| in registry:", "my-lab-edge5" in profiles.PROFILES)
print("usable under the v3.5 engine:", profiles.is_v35_flavoured(registered))
try:
    profiles.register_profile(mine)
except ValueError as err:
    print("second registration refused:", str(err).split(";")[0])

registered: my-lab-edge5 | in registry: True
usable under the v3.5 engine: False
second registration refused: fidelity profile 'my-lab-edge5' is already registered


## Select it by name from a configuration file

A run configuration names its profile under `[fidelity].profile`. `config.load_config` resolves
the name against the registry at load time, so a profile registered in this process is selectable
and the loaded configuration holds the very object that was registered. In a separate process,
such as the command-line driver or its worker processes, the registration has to run there too:
`[fidelity].profile_module` names a Python module that the loader imports first, and that module
calls `register_profile()`.

In [5]:
import tempfile
from pathlib import Path
from arrayprobe import config

work = Path(tempfile.mkdtemp(prefix="nb41_"))
selects = work / "select.toml"
selects.write_text('[fidelity]\nprofile = "my-lab-edge5"\n')
print(selects.read_text())
cfg = config.load_config(selects)
print("selected:", cfg.profile.name, "| same object as registered:", cfg.profile is registered)
print("its max_edge_img_ar:", cfg.profile.reject_thresholds.max_edge_img_ar)

[fidelity]
profile = "my-lab-edge5"

selected: my-lab-edge5 | same object as registered: True
its max_edge_img_ar: 5


## A configuration file that tries to set a threshold is refused

Writing the threshold itself into the TOML file, instead of naming a profile that carries it, is
refused by `load_config` with a `ConfigError`. Every problem in the file is reported at once, on
`.problems`. A name nothing has registered is refused the same way.

In [6]:
describes = work / "describe.toml"
describes.write_text('[fidelity]\nprofile = "stock-v3.5"\nmax_edge_img_ar = 5\n')
print(describes.read_text())
try:
    config.load_config(describes)
except config.ConfigError as err:
    for problem in err.problems:
        print("refused:", problem.split(" (one of")[0] + ", never by a raw key")

unknown = work / "unknown.toml"
unknown.write_text('[fidelity]\nprofile = "my-lab-edge4"\n')
try:
    config.load_config(unknown)
except config.ConfigError as err:
    print("refused:", str(err.problems[0]).split(";")[0])

[fidelity]
profile = "stock-v3.5"
max_edge_img_ar = 5

refused: [fidelity] sets ['max_edge_img_ar']: fidelity thresholds are chosen by [fidelity].profile, never by a raw key
refused: unknown profile 'my-lab-edge4'


## How the chosen profile is recorded

Every product the run driver writes gets a `.call` sidecar naming the engine and the profile that
made it, and a later run refuses an input whose stamp disagrees with its own configuration. A run
is not executed here, so the stamp is written with the public writer that shares the driver's
format, `arrayprobe_uiops.provenance.write`, on a placeholder file, with the selected profile's
reject thresholds beside its name in the driver's `reject=(...)` form. `provenance.read` reads it
back, and `attributed_export.parse_reject` splits the thresholds into keys.

In [7]:
from arrayprobe_uiops import provenance
from arrayprobe import attributed_export as ae

rc = cfg.profile.reject_thresholds
reject = "(" + ",".join(f"{f.name}={getattr(rc, f.name)}" for f in dataclasses.fields(rc)) + ")"
product = work / "proc.example.nc"
product.write_bytes(b"placeholder")
sidecar = provenance.write(product, engine="3.5", profile=cfg.profile.name,
                           extra={"reject": reject})
print(sidecar.name)
print(sidecar.read_text().strip())

stamp = provenance.read(product)
print("engine:", stamp.engine, "| profile:", stamp.profile)
print("max_edge_img_ar recorded:", ae.parse_reject(stamp.fields["reject"])["max_edge_img_ar"])

proc.example.nc.call
engine=3.5 profile=my-lab-edge5 reject=(rej_zero_area=True,rej_split=True,rej_hollow=True,shadowed_area=25,max_edge_img_ar=5,max_comp_img_ar=5)
engine: 3.5 | profile: my-lab-edge5
max_edge_img_ar recorded: 5


**What has no public API here.** The run driver composes its full stamp (which also names the
shadow-diode handling, any aircraft TAS file and where each threshold came from) from its own run
state while a run executes, so that exact text appears only from a driver run on raw data, which
this notebook does not do.

## Summary

- `profiles.SHIPPED` and `profiles.PROFILES` list the shipped and the registered profiles;
  `profiles.profile(name)` returns one.
- A variant is a `dataclasses.replace` of a shipped profile with its own `name` and `description`,
  added with `profiles.register_profile()`; a name registers once.
- A TOML file selects a registered profile with `[fidelity].profile`; `config.load_config` refuses
  a raw threshold in the file and an unregistered name. `[fidelity].profile_module` makes a
  registration reachable from another process.
- The profile's name, beside the engine, is what a product's `.call` stamp records, read back with
  `provenance.read`.